# Chapter 9 — Real-Time Webcam Capture
**Project:** PTVNDM  
**Notebook:** `notebooks/01_webcam_capture.ipynb`

This notebook captures live video from the default webcam and saves it to `../outputs/captured_video.avi`.

**Controls:**
- Press **`q`** in the *'Video Capture'* window to stop recording and release all resources.

## 1 · Imports

In [ ]:
import cv2
import os

print(f"OpenCV version: {cv2.__version__}")

## 2 · Configuration

In [ ]:
# ── Output settings ────────────────────────────────────────────────────────────
OUTPUT_DIR  = "../outputs"          # relative to notebooks/
OUTPUT_FILE = "captured_video.avi"
OUTPUT_PATH = os.path.join(OUTPUT_DIR, OUTPUT_FILE)

# ── Video writer settings ──────────────────────────────────────────────────────
FOURCC      = cv2.VideoWriter_fourcc(*"XVID")  # XVID codec → .avi
FPS         = 20.0
RESOLUTION  = (640, 480)            # (width, height)

# ── Webcam settings ────────────────────────────────────────────────────────────
CAMERA_INDEX   = 0                  # default webcam
WINDOW_NAME    = "Video Capture"
QUIT_KEY       = ord("q")           # press 'q' to stop

# Ensure the output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output will be saved to: {os.path.abspath(OUTPUT_PATH)}")

## 3 · Capture & Record

> **Stop recording:** press **`q`** in the *'Video Capture'* window.

In [ ]:
# ── 1. Open webcam ─────────────────────────────────────────────────────────────
cap = cv2.VideoCapture(CAMERA_INDEX)

if not cap.isOpened():
    raise IOError(
        f"Cannot open webcam (camera index {CAMERA_INDEX}). "
        "Check that your webcam is connected and not used by another application."
    )

print("Webcam opened successfully.")

# ── 2. Create VideoWriter ──────────────────────────────────────────────────────
out = cv2.VideoWriter(OUTPUT_PATH, FOURCC, FPS, RESOLUTION)

if not out.isOpened():
    cap.release()
    raise IOError(
        f"Cannot create VideoWriter for path '{OUTPUT_PATH}'. "
        "Check codec support and output directory permissions."
    )

print(f"Recording started → {os.path.abspath(OUTPUT_PATH)}")
print("Press 'q' in the 'Video Capture' window to stop.")

# ── 3. Capture loop ────────────────────────────────────────────────────────────
try:
    while True:
        ret, frame = cap.read()

        if not ret:
            print("Warning: Failed to grab a frame. Stopping capture.")
            break

        # Resize frame to the target resolution before writing
        frame_resized = cv2.resize(frame, RESOLUTION)

        # Write the frame to the output video file
        out.write(frame_resized)

        # Display the live feed
        cv2.imshow(WINDOW_NAME, frame_resized)

        # Break on 'q' key (wait 1 ms between frames)
        if cv2.waitKey(1) & 0xFF == QUIT_KEY:
            print("'q' pressed — stopping recording.")
            break

finally:
    # ── 4. Release all resources ───────────────────────────────────────────────
    cap.release()
    out.release()
    cv2.destroyAllWindows()
    print("Resources released. All OpenCV windows closed.")
    print(f"Video saved to: {os.path.abspath(OUTPUT_PATH)}")

## 4 · Verify Output

In [ ]:
if os.path.exists(OUTPUT_PATH):
    size_mb = os.path.getsize(OUTPUT_PATH) / (1024 ** 2)
    print(f"✔ File found: {os.path.abspath(OUTPUT_PATH)}")
    print(f"  Size: {size_mb:.2f} MB")

    # Open the saved file with OpenCV to confirm readability
    verify = cv2.VideoCapture(OUTPUT_PATH)
    if verify.isOpened():
        total_frames = int(verify.get(cv2.CAP_PROP_FRAME_COUNT))
        saved_fps    = verify.get(cv2.CAP_PROP_FPS)
        width        = int(verify.get(cv2.CAP_PROP_FRAME_WIDTH))
        height       = int(verify.get(cv2.CAP_PROP_FRAME_HEIGHT))
        duration_s   = total_frames / saved_fps if saved_fps > 0 else 0

        print(f"  Resolution : {width}x{height}")
        print(f"  FPS        : {saved_fps}")
        print(f"  Frames     : {total_frames}")
        print(f"  Duration   : {duration_s:.1f} seconds")
        verify.release()
    else:
        print("Warning: File exists but could not be opened by OpenCV for verification.")
else:
    print(f"✘ File not found at: {os.path.abspath(OUTPUT_PATH)}")